# Analyse de performance — Partie 4 (optimisation)

Démarche : profiler `/predict` pour trouver le vrai goulot d'étranglement (pas deviner),
tester une piste d'optimisation informée par les résultats, vérifier qu'elle n'introduit
aucune régression, mesurer le gain réel de bout en bout, puis intégrer dans l'API.

*Notebook généré avec l'assistance de Claude (Anthropic), en accompagnement du développement du projet.*

In [1]:
import os
import sys
import time
import cProfile
import pstats

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.path.join(os.getcwd(), "app"))

import joblib
import numpy as np
import onnxruntime as ort
from catboost import Pool
from dotenv import load_dotenv

load_dotenv()

import main
from customer import get_customer
from explain import get_top_influential_features, FEATURES_INTERPRETABLES

# Reproduit ce que fait lifespan() au démarrage de l'API.
pipeline, seuil_optimal, score = joblib.load("model.pkl")
modele = pipeline.named_steps["model"]

CUSTOMER_ID = 100002
customer_df = get_customer(CUSTOMER_ID).drop(columns=["TARGET"], errors="ignore")
noms_colonnes = list(customer_df.columns)

## 1. Profiling — où passe le temps réellement ?

`cProfile` sur le flux d'origine (`predict_proba` CatBoost + calcul SHAP), avant toute
optimisation.

In [2]:
N_PROFIL = 200


def flux_origine():
    proba = pipeline.predict_proba(customer_df)[:, 1]
    facteurs = get_top_influential_features(modele, customer_df, candidats=FEATURES_INTERPRETABLES)
    return proba, facteurs


profiler = cProfile.Profile()
profiler.enable()
for _ in range(N_PROFIL):
    flux_origine()
profiler.disable()

print(f"--- Profil sur {N_PROFIL} appels (flux d'origine) ---\n")
stats = pstats.Stats(profiler).sort_stats("cumulative")
stats.print_stats(10)

--- Profil sur 200 appels (flux d'origine) ---

         14086926 function calls (12567605 primitive calls) in 10.360 seconds

   Ordered by: cumulative time
   List reduced from 536 to 10 due to restriction <10>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      200    0.001    0.000    4.402    0.022 /home/helbram/.local/lib/python3.14/site-packages/sklearn/pipeline.py:815(predict_proba)
      200    0.018    0.000    4.367    0.022 /home/helbram/.local/lib/python3.14/site-packages/catboost/core.py:5611(predict_proba)
      200    0.001    0.000    4.277    0.021 /home/helbram/.local/lib/python3.14/site-packages/catboost/core.py:2922(_predict)
      200    0.002    0.000    4.200    0.021 /home/helbram/.local/lib/python3.14/site-packages/catboost/core.py:2900(_process_predict_input_data)
      200    0.023    0.000    1.523    0.008 /home/helbram/.local/lib/python3.14/site-packages/catboost/core.py:3385(get_feature_importance)
   505600    0.667    0.000  

**FAITS** : la quasi-totalité du temps cumulé est dans `catboost/core.py:_init` — la
construction de l'objet `Pool` à partir du DataFrame pandas (1264 colonnes) — appelée
**deux fois** par prédiction (une fois pour `predict_proba`, une fois pour le SHAP), pas
le calcul du modèle lui-même.

**INTERPRÉTATION** : optimiser "le modèle" n'a pas de sens ici — il faut réduire le coût
de conversion DataFrame → format CatBoost, qui domine largement le temps de calcul réel.

## 2. Piste 1 — ONNX Runtime pour la prédiction

Export du modèle CatBoost en ONNX, puis comparaison à l'implémentation native — en
vérifiant d'abord qu'il n'y a **aucune régression** de précision avant de comparer quoi
que ce soit en vitesse.

In [3]:
modele.save_model("model.onnx", format="onnx")
session = ort.InferenceSession("model.onnx")
entree_nom = session.get_inputs()[0].name


def predire_catboost():
    return pipeline.predict_proba(customer_df)[0, 1]


def predire_onnx():
    entree = customer_df.to_numpy(dtype=np.float32)
    return session.run(None, {entree_nom: entree})[1][0][1]


# --- Non-régression AVANT toute comparaison de temps ---
proba_catboost = predire_catboost()
proba_onnx = predire_onnx()
ecart = abs(proba_catboost - proba_onnx)
print(f"CatBoost : {proba_catboost:.6f}")
print(f"ONNX     : {proba_onnx:.6f}")
print(f"Écart    : {ecart:.8f}")
assert ecart < 0.01, "Écart de prédiction trop important entre CatBoost et ONNX"
print("OK — prédictions équivalentes\n")


def chronometrer(fn, n=500):
    debut = time.perf_counter()
    for _ in range(n):
        fn()
    return (time.perf_counter() - debut) / n * 1000


ms_catboost = chronometrer(predire_catboost)
ms_onnx = chronometrer(predire_onnx)
print(f"CatBoost natif (predict_proba seul) : {ms_catboost:.2f} ms/appel")
print(f"ONNX Runtime (predict seul)         : {ms_onnx:.2f} ms/appel")
print(f"Gain (prédiction isolée)            : {(1 - ms_onnx / ms_catboost) * 100:.1f}%")

CatBoost : 0.286347
ONNX     : 0.286347
Écart    : 0.00000001
OK — prédictions équivalentes



CatBoost natif (predict_proba seul) : 11.90 ms/appel
ONNX Runtime (predict seul)         : 0.05 ms/appel
Gain (prédiction isolée)            : 99.6%


**Limite importante** : ce gain isolé sur la prédiction ne se traduit pas directement
sur `/predict` complet, car ONNX ne fournit pas de calcul SHAP (explicabilité) — le
modèle CatBoost natif reste nécessaire pour les facteurs influents. Le gain de bout en
bout, mesuré ci-dessous, sera donc significativement plus faible que ce chiffre isolé.

In [4]:
def flux_onnx_seul():
    proba = predire_onnx()
    facteurs = get_top_influential_features(modele, customer_df, candidats=FEATURES_INTERPRETABLES)
    return proba, facteurs


ms_flux_origine = chronometrer(flux_origine, n=200)
ms_flux_onnx_seul = chronometrer(flux_onnx_seul, n=200)

print(f"Flux d'origine (CatBoost predict + CatBoost SHAP) : {ms_flux_origine:.2f} ms/appel")
print(f"Flux ONNX seul (ONNX predict + CatBoost SHAP)     : {ms_flux_onnx_seul:.2f} ms/appel")
print(f"Gain de bout en bout                              : {(1 - ms_flux_onnx_seul / ms_flux_origine) * 100:.1f}%")

Flux d'origine (CatBoost predict + CatBoost SHAP) : 30.80 ms/appel
Flux ONNX seul (ONNX predict + CatBoost SHAP)     : 19.72 ms/appel
Gain de bout en bout                              : 36.0%


## 3. Piste 2 — Un seul `Pool`, construit depuis un array plutôt qu'un DataFrame

Le profiling (section 1) pointe vers la construction du `Pool` elle-même, pas seulement
sa duplication. Hypothèse : `Pool(DataFrame)` est lent parce que pandas doit inspecter
le type de chaque colonne une par une ; `Pool(array numpy)` évite ce scan, le tableau
étant déjà homogène.

In [5]:
arr = customer_df.to_numpy(dtype=np.float32)

ms_pool_df = chronometrer(lambda: Pool(customer_df), n=500)
ms_pool_arr = chronometrer(lambda: Pool(arr), n=500)

print(f"Pool(DataFrame)    : {ms_pool_df:.2f} ms")
print(f"Pool(array numpy)  : {ms_pool_arr:.2f} ms")
print(f"Gain               : {(1 - ms_pool_arr / ms_pool_df) * 100:.1f}%")

Pool(DataFrame)    : 11.08 ms
Pool(array numpy)  : 1.82 ms
Gain               : 83.6%


Hypothèse confirmée. On combine les deux pistes : prédiction via ONNX (sur l'array),
et un **seul** `Pool` (construit depuis ce même array, pas depuis le DataFrame) réutilisé
pour le calcul SHAP — au lieu d'en reconstruire un deuxième.

In [6]:
def flux_optimise():
    arr = customer_df.to_numpy(dtype=np.float32)
    proba = session.run(None, {entree_nom: arr})[1][0][1]
    pool = Pool(arr, feature_names=noms_colonnes)
    facteurs = get_top_influential_features(modele, customer_df, candidats=FEATURES_INTERPRETABLES, pool=pool)
    return proba, facteurs


# --- Non-régression : les valeurs SHAP doivent être identiques à celles du flux d'origine ---
pool_df = Pool(customer_df)
shap_origine = modele.get_feature_importance(pool_df, type="ShapValues")[0]
pool_arr = Pool(arr, feature_names=noms_colonnes)
shap_optimise = modele.get_feature_importance(pool_arr, type="ShapValues")[0]
ecart_shap = np.abs(np.array(shap_origine) - np.array(shap_optimise)).max()
print(f"Écart max valeurs SHAP (array vs DataFrame) : {ecart_shap:.8f}")
assert ecart_shap < 1e-4, "Les valeurs SHAP diffèrent entre les deux méthodes de construction du Pool"
print("OK — facteurs influents identiques\n")

ms_flux_optimise = chronometrer(flux_optimise, n=200)

print(f"Flux d'origine  : {ms_flux_origine:.2f} ms/appel")
print(f"Flux ONNX seul  : {ms_flux_onnx_seul:.2f} ms/appel")
print(f"Flux optimisé   : {ms_flux_optimise:.2f} ms/appel  (ONNX + Pool partagé depuis array)")
print(f"\nGain final de bout en bout : {(1 - ms_flux_optimise / ms_flux_origine) * 100:.1f}%")

Écart max valeurs SHAP (array vs DataFrame) : 0.00000000
OK — facteurs influents identiques



Flux d'origine  : 30.80 ms/appel
Flux ONNX seul  : 19.72 ms/appel
Flux optimisé   : 11.32 ms/appel  (ONNX + Pool partagé depuis array)

Gain final de bout en bout : 63.2%


## 4. Intégration dans l'API

Changements apportés (voir aussi le code, ce sont les vraies modifications) :
- `app/model.py` : exporte `model.onnx` en plus de `model.pkl` à la fin de l'entraînement.
- `app/explain.py` : `get_top_influential_features()` accepte un `Pool` déjà construit
  (paramètre `pool`), au lieu d'en reconstruire un systématiquement.
- `app/main.py` : `lifespan` charge la session ONNX en plus du pickle CatBoost.
  `run_prediction()` utilise ONNX pour la prédiction, construit un seul `Pool` depuis
  l'array numpy, le réutilise pour le SHAP. `duree_ms` mesure le temps de bout en bout
  (prédiction + SHAP), pas juste la prédiction — cohérent avec ce qui est comparé ici.
- `Dockerfile` / `requirements.txt` : `model.onnx` ajouté à l'image, `onnxruntime` ajouté
  aux dépendances de production (pas seulement dev — l'API en dépend directement).

Vérifié : les 16 tests automatisés passent sans changement de comportement, et un appel
réel à `/predict/100002` renvoie exactement la même prédiction et les mêmes facteurs
influents qu'avant l'optimisation (aucune régression fonctionnelle).

## 5. Bilan

**FAITS** :
- Goulot d'étranglement identifié par profiling : construction du `Pool` CatBoost depuis
  un DataFrame large (1264 colonnes), faite deux fois par requête — pas le calcul du
  modèle lui-même.
- ONNX seul : gain isolé énorme sur la prédiction (~99%), mais limité de bout en bout
  (~29%) car il ne couvre pas le calcul SHAP.
- Construire le `Pool` depuis un array plutôt qu'un DataFrame, et le partager entre
  prédiction et SHAP : ~84% de gain sur la construction du `Pool` lui-même.
- Combiné (ONNX + `Pool` partagé) : gain de bout en bout mesuré ci-dessus, vérifié sans
  régression de précision (prédiction et valeurs SHAP identiques à la version d'origine).

**INTERPRÉTATION** : le vrai levier n'était pas "un modèle plus rapide", mais éliminer
un travail de conversion de données redondant. C'est un exemple concret de l'intérêt du
profiling avant optimisation — la solution qui a le plus rapporté (Pool partagé, 84% sur
sa propre étape) n'était pas la piste initialement demandée (ONNX), mais une découverte
faite en creusant les données de profiling.

**CE QUI RESTE À VÉRIFIER** : le premier appel après démarrage de l'API montre un léger
surcoût ("cold start" ONNX Runtime, session pas encore "chaude") — non quantifié
précisément ici, à surveiller via `scripts/analyse_operationnel.py` une fois l'API en
usage réel.